# Comparación offline de SigLIP y SigLIP2

Se comparan los checkpoints base de resolución fija 224 bajo el mismo protocolo, con idénticas imágenes, consultas, preprocesado y métricas. El conjunto propio actúa como verificación funcional; SUN RGB-D es el conjunto discriminativo. Los intervalos se obtienen mediante bootstrap por consulta.

In [1]:
from pathlib import Path
import sys
repo = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'semantic_navigation_ws' / 'src').is_dir())
sys.path.insert(0, str(repo / 'experiments' / 'shared'))
import pandas as pd
from notebook_bootstrap import bootstrap_offline, resolve_repo_path
from offline_benchmarks import create_vlm_figures, run_vlm_benchmark
ctx = bootstrap_offline()
print(f"Dispositivo: {ctx['device']}")
display(pd.DataFrame([{'modelo': key, 'checkpoint': value}
                      for key, value in ctx['config']['models']['siglip']['variants'].items()]))

/home/junior/visual_semantic_navigation/.venv-1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dispositivo: cuda


,modelo,checkpoint
0,siglip_v1,google/siglip-base-patch16-224
1,siglip_v2,google/siglip2-base-patch16-224


## Ejecución y resultados numéricos

In [2]:
results = run_vlm_benchmark(ctx)
cases = results['cases']
summary = results['summary']
display(summary)
display(results['paired_differences'])
display(results['threshold_diagnostics'])
display(results['model_costs'])

Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 408/408 [00:00<00:00, 5038.78it/s]

Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 408/408 [00:00<00:00, 5544.61it/s]

,dataset_id,method,query_type,language,n_queries,n_positive,n_negative,recall_at_1,recall_at_1_ci_low,recall_at_1_ci_high,...,recall_at_5,recall_at_5_ci_low,recall_at_5_ci_high,mean_reciprocal_rank,mean_reciprocal_rank_ci_low,mean_reciprocal_rank_ci_high,negative_rejection_rate,negative_rejection_rate_ci_low,negative_rejection_rate_ci_high,mean_retrieval_latency_ms
0,siglip_rooms,random_baseline,attribute,es,1,1,0,0.0,NaN,NaN,...,0.0,NaN,NaN,0.166667,NaN,NaN,NaN,NaN,NaN,0.202196
1,siglip_rooms,random_baseline,functional,en,1,1,0,0.0,NaN,NaN,...,1.0,NaN,NaN,0.500000,NaN,NaN,NaN,NaN,NaN,0.202404
2,siglip_rooms,random_baseline,functional,es,1,1,0,1.0,NaN,NaN,...,1.0,NaN,NaN,1.000000,NaN,NaN,NaN,NaN,NaN,0.216025
3,siglip_rooms,random_baseline,multi_object,en,1,1,0,0.0,NaN,NaN,...,1.0,NaN,NaN,0.333333,NaN,NaN,NaN,NaN,NaN,0.210297
4,siglip_rooms,random_baseline,multi_object,es,1,1,0,0.0,NaN,NaN,...,1.0,NaN,NaN,0.500000,NaN,NaN,NaN,NaN,NaN,0.199192
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79,sunrgbd,siglip_v2,negative,es,4,0,4,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,32.816114
80,sunrgbd,siglip_v2,object,en,6,6,0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.006628,0.002779,0.010831,NaN,NaN,NaN,33.101137
81,sunrgbd,siglip_v2,object,es,6,6,0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.006391,0.002697,0.010665,NaN,NaN,NaN,32.477941
82,sunrgbd,siglip_v2,room,en,13,13,0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.017048,0.008549,0.026898,NaN,NaN,NaN,32.738547


,dataset_id,n_paired_queries,delta_recall_at_1,delta_recall_at_1_ci_low,delta_recall_at_1_ci_high,delta_reciprocal_rank,delta_reciprocal_rank_ci_low,delta_reciprocal_rank_ci_high
0,siglip_rooms,11,-0.909091,-1.00,-0.727273,-0.730501,-0.878235,-0.535982
1,sunrgbd,50,-0.600000,-0.72,-0.460000,-0.707600,-0.800536,-0.611790


,dataset_id,method,descriptive_threshold,balanced_accuracy,positive_acceptance,negative_rejection,n_positive,n_negative,warning
0,siglip_rooms,siglip_v1,0.054272,1.000000,1.000000,1.000,11,2,descriptive only; estimated and evaluated on t...
1,siglip_rooms,siglip_v2,0.051546,0.613636,0.727273,0.500,11,2,descriptive only; estimated and evaluated on t...
2,sunrgbd,siglip_v1,0.073337,0.937500,1.000000,0.875,50,8,descriptive only; estimated and evaluated on t...
3,sunrgbd,siglip_v2,0.081027,0.550000,0.600000,0.500,50,8,descriptive only; estimated and evaluated on t...


,method,model_id,model_load_s,device,parameters,parameter_memory_mb,embedding_dimension,mean_image_encoding_ms,std_image_encoding_ms,mean_text_encoding_ms,std_text_encoding_ms,benchmark_samples
0,siglip_v1,google/siglip-base-patch16-224,0.717142,cuda,203155970,774.978523,768,20.169008,1.345765,20.478244,49.927096,20
1,siglip_v2,google/siglip2-base-patch16-224,5.486709,cuda,375187970,1431.228523,768,19.724084,0.681976,8.714183,0.500637,20


## Figuras

Se exportan en PNG para inspección y en PDF vectorial para su incorporación a la memoria.

In [3]:
from reproducibility import collect_manifest, save_manifest
results_root = resolve_repo_path(ctx['repo_root'], ctx['config']['paths']['results_root']) / 'vlm_comparison'
figures_root = results_root / 'figures'
results_root.mkdir(parents=True, exist_ok=True)
for name, frame in results.items():
    frame.to_csv(results_root / f'{name}.csv', index=False)
figure_paths = create_vlm_figures(cases, results['model_costs'], figures_root)
manifest = collect_manifest(ctx['config'], repo_dir=str(ctx['repo_root']), device=ctx['device'],
    extra={'notebook': '01_siglip_retrieval', 'n_cases': len(cases),
           'models': ctx['config']['models']['siglip']['variants'],
           'figures': figure_paths})
save_manifest(str(results_root / 'manifest.json'), manifest)
print(f'Resultados: {results_root}')
print('Figuras generadas:')
for path in figure_paths:
    print(' -', path)

Resultados: /home/junior/visual_semantic_navigation/experiments/offline/results/vlm_comparison
Figuras generadas:
 - /home/junior/visual_semantic_navigation/experiments/offline/results/vlm_comparison/figures/vlm_recall_por_tipo.png
 - /home/junior/visual_semantic_navigation/experiments/offline/results/vlm_comparison/figures/vlm_recall_por_tipo.pdf
 - /home/junior/visual_semantic_navigation/experiments/offline/results/vlm_comparison/figures/vlm_recall_at_k.png
 - /home/junior/visual_semantic_navigation/experiments/offline/results/vlm_comparison/figures/vlm_recall_at_k.pdf
 - /home/junior/visual_semantic_navigation/experiments/offline/results/vlm_comparison/figures/vlm_comparacion_idiomas.png
 - /home/junior/visual_semantic_navigation/experiments/offline/results/vlm_comparison/figures/vlm_comparacion_idiomas.pdf
 - /home/junior/visual_semantic_navigation/experiments/offline/results/vlm_comparison/figures/vlm_coste_inferencia.png
 - /home/junior/visual_semantic_navigation/experiments/offl

## Lectura automática de control

In [4]:
sun = cases.loc[(cases['dataset_id'] == 'sunrgbd') & (~cases['is_negative'])
                & cases['method'].isin(['siglip_v1', 'siglip_v2'])]
control = sun.groupby('method')[['recall_at_1', 'recall_at_3', 'recall_at_5', 'reciprocal_rank']].mean()
display(control)
winner = control['recall_at_1'].idxmax()
print(f'Mayor Recall@1 descriptivo sobre SUN RGB-D: {winner} ({control.loc[winner, "recall_at_1"]:.3f}).')
print('La selección definitiva requiere considerar también coste e intervalos, no solo esta media.')

,recall_at_1,recall_at_3,recall_at_5,reciprocal_rank
method,,,,
siglip_v1,0.6,0.84,0.92,0.734556
siglip_v2,0.0,0.04,0.04,0.026955


Mayor Recall@1 descriptivo sobre SUN RGB-D: siglip_v1 (0.600).
La selección definitiva requiere considerar también coste e intervalos, no solo esta media.
